# Tutorial n°4

## Temperature evolution of the electronic spectra

### A. The cationic Methylene-Pyrene Isomers

In [ ]:
import deMonPy

deMonPy.configure_from_file("../global.json")

In [ ]:
from deMonPy.molden import read_XYZ

images, comm = read_XYZ("./data/methylpyren_1.xyz")
mpyren_1 = images[-1]

In [ ]:
from deMonPy.deMonNano import deMonNano

WORKDIR = ".run/tutorial-4/A/"

parameters_opt = {
    "DEMON_EXECUTABLE": deMonPy.DEMON_EXECUTABLE,
    "BASIS": {"PTYPE": "BIO", "SKFILE": deMonPy.DEMON_BASIS},
    "DEMON_PARAMETERS": {
        "ACTIVE": {"DFTB": {"SCC": True}, "CHARGE": 1.0},
    },
    "DEMON_MODULE": {"ACTIVE": {"OPT": {"MAX": 9999, "OUT": 1, "TRAJECTORY": True}}},
}

dem = deMonNano(title="CALCULATION DEMONANO", workdir=WORKDIR, **parameters_opt)

dem.calculate(symbols=mpyren_1.symbols, positions=mpyren_1.positions)

In [ ]:
from ase.visualize import view

view(dem.results["trajectory"])

### B. The TD-DFTB

In [ ]:
optimized_geometry_1 = dem.results["output_geometry"]
print("Optimized geometry : ", optimized_geometry_1)

In [ ]:
from deMonPy.deMonNano import deMonNano

WORKDIR = ".run/tutorial-4/B/"

parameters_lresp = {
    "DEMON_EXECUTABLE": deMonPy.DEMON_EXECUTABLE,
    "BASIS": {"PTYPE": "BIO", "SKFILE": deMonPy.DEMON_BASIS},
    "DEMON_PARAMETERS": {
        "ACTIVE": {"DFTB": {"SCC": True}, "CHARGE": 1.0, "TD-DFTB": True},
    },
}

dem = deMonNano(title="CALCULATION DEMONANO", workdir=WORKDIR, **parameters_lresp)

dem.calculate(
    symbols=optimized_geometry_1.symbols, positions=optimized_geometry_1.positions
)

In [ ]:
results = dem.results

iso1_singlet = results["singlet"]
iso1_triplet = results["triplet"]

### C. Comparizon with isomer 2

In [ ]:
from deMonPy.molden import read_XYZ

images, comm = read_XYZ("./data/methylpyren_2.xyz")
mpyren_2 = images[-1]

In [ ]:
from deMonPy.deMonNano import deMonNano

WORKDIR = ".run/tutorial-4/C/"

dem = deMonNano(title="CALCULATION DEMONANO", workdir=WORKDIR, **parameters_opt)

dem.calculate(symbols=mpyren_2.symbols, positions=mpyren_2.positions)

optimized_geometry_2 = dem.results["output_geometry"]

In [ ]:
dem = deMonNano(title="CALCULATION DEMONANO", workdir=WORKDIR, **parameters_lresp)

dem.calculate(
    symbols=optimized_geometry_2.symbols, positions=optimized_geometry_2.positions
)

results = dem.results

iso2_singlet = results["singlet"]
iso2_triplet = results["triplet"]

In [ ]:
import numpy as np


def transform_to_table(data):
    table = []

    for elm in ["w", "oscillator", "weight", "energy"]:
        table.append([d[elm] for k, d in data.items()])
    return np.array(table).T


tab1 = transform_to_table(iso1_singlet)
tab2 = transform_to_table(iso2_singlet)

print("Singlet table in isomer 1 : \n", tab1)
print("Singlet table in isomer 2 : \n", tab2)

### D. Temperature evolution 

In [ ]:
print(optimized_geometry_1)

parameters_mdlresp = {
    "DEMON_EXECUTABLE": deMonPy.DEMON_EXECUTABLE,
    "BASIS": {"PTYPE": "BIO", "SKFILE": deMonPy.DEMON_BASIS},
    "DEMON_PARAMETERS": {
        "ACTIVE": {"DFTB": {"SCC": True}, "CHARGE": 1, "TD-DFTB": {"FRESP": 10}},
    },
    "DEMON_MODULE": {
        "ACTIVE": {
            "MD": {
                "MDYNAMICS": {
                    "RANDOM": 300,
                },
                "TIMESTEP": 0.5,
                "MDSTEP": {"MAX": 100, "OUT": 1},
                "TRAJECTORY": True,
            }
        }
    },
}

In [ ]:
from deMonPy.deMonNano import deMonNano

WORKDIR = ".run/tutorial-4/D/"

dem = deMonNano(title="CALCULATION DEMONANO", workdir=WORKDIR, **parameters_mdlresp)

dem.calculate(
    symbols=optimized_geometry_1.symbols, positions=optimized_geometry_1.positions
)

results = dem.results

In [ ]:
import os

import matplotlib.pyplot as plt

src = os.path.join(WORKDIR, "spectrum_final.out")

with open(src, "r") as fd:
    lines = fd.readlines()

data = np.array([[float(line.split()[0]), float(line.split()[1])] for line in lines])

plt.plot(data[:, 0], data[:, 1])
plt.show()